# CDIME-AI — Self-Contained Colab Notebook (Free T4)
**Causal Domain-Incremental Multimodal Explainable AI for Chest X-ray Diagnosis & Therapeutic Decision Support**

This single notebook is **fully self-contained** — no git clone, no file uploads.
Running the cells in order recreates the entire `cdime_ai` framework on the Colab
filesystem and runs the complete experiment: continual learning across
CheXpert → MIMIC-CXR → VinDr-CXR, causal (IRM + V-REx) regularisation,
EWC + replay + domain adapters, Grad-CAM / attention / counterfactual
explanations, MC-Dropout + temperature-scaling uncertainty, therapeutic decision
support, ablations (A1–A7) and statistical significance tests.

### How to use
1. **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.
2. By default it uses a **causal synthetic data generator** (no downloads) so you
   get a full results table in minutes. To use the real datasets, see the last
   section.

> Decision-support research software only — **not** a medical device.

## 0. Install dependencies

In [ ]:
!pip install -q timm transformers scikit-learn scipy >/dev/null 2>&1
import torch
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available(),
      '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (set runtime to T4!)')

## 1. Build the `cdime_ai` package
Each cell below writes one module to disk with `%%writefile`. Run them all once;
they create the package directory tree that the experiment cells import.

In [ ]:
import os
os.makedirs('cdime_ai/causal', exist_ok=True)
os.makedirs('cdime_ai/continual', exist_ok=True)
os.makedirs('cdime_ai/data', exist_ok=True)
os.makedirs('cdime_ai/decision', exist_ok=True)
os.makedirs('cdime_ai/engine', exist_ok=True)
os.makedirs('cdime_ai/evaluation', exist_ok=True)
os.makedirs('cdime_ai/explain', exist_ok=True)
os.makedirs('cdime_ai/models', exist_ok=True)
os.makedirs('cdime_ai/uncertainty', exist_ok=True)
os.makedirs('cdime_ai', exist_ok=True)
print('package dirs ready')

In [ ]:
%%writefile cdime_ai/__init__.py
"""CDIME-AI: Causal Domain-Incremental Multimodal Explainable AI for CXR.

A reference implementation of the CDIME-AI framework: a multimodal
(image + radiology report) classifier that learns causal, domain-invariant
representations, adapts continually across hospitals without catastrophic
forgetting, explains its predictions, quantifies uncertainty and produces
evidence-based therapeutic decision support.

Quick start
-----------
>>> from cdime_ai.config import Config
>>> from cdime_ai.tokenizer import get_tokenizer
>>> from cdime_ai.data.loaders import build_domain_loaders
>>> from cdime_ai.engine.continual_runner import run_continual
>>> cfg = Config.fast_debug()
>>> tok = get_tokenizer(cfg)
>>> loaders = build_domain_loaders(cfg, tok)
>>> results = run_continual(cfg, loaders, tok)
"""
from .config import Config, LABELS

__version__ = "1.0.0"
__all__ = ["Config", "LABELS"]


In [ ]:
%%writefile cdime_ai/config.py
"""Central configuration for the CDIME-AI framework.

All defaults are tuned for a free Google Colab T4 GPU (~16 GB VRAM) with low
host RAM. Increase the values marked ``# scale-up`` when running on bigger
hardware to reproduce full-scale Q1 results.
"""
from __future__ import annotations

from dataclasses import dataclass, field, asdict
from typing import List, Optional
import json
import torch


# The five canonical CheXpert competition pathologies. These overlap across
# CheXpert, MIMIC-CXR and VinDr-CXR which makes them ideal for the
# domain-incremental protocol (same label space, shifting input distribution).
LABELS: List[str] = [
    "Atelectasis",
    "Cardiomegaly",
    "Consolidation",
    "Edema",
    "Pleural Effusion",
]


@dataclass
class DataConfig:
    """Data-loading configuration."""

    # "synthetic" runs the full pipeline end-to-end with no downloads (default,
    # T4-friendly). "real" uses the on-disk CheXpert / MIMIC-CXR / VinDr-CXR
    # CSVs configured in data/datasets.py.
    mode: str = "synthetic"

    image_size: int = 224
    max_text_len: int = 96            # short radiology impressions; keeps BERT cheap

    # Per-domain sample counts. Tiny by default so a full 3-domain continual run
    # finishes in minutes on a T4. # scale-up
    train_per_domain: int = 600
    val_per_domain: int = 150
    test_per_domain: int = 200

    num_workers: int = 2              # low for limited Colab RAM
    pin_memory: bool = True

    # Strength of the domain-dependent spurious shortcut injected by the
    # synthetic generator (0 = none, 1 = perfectly predictive shortcut).
    # This is what lets us *demonstrate* the causal module's benefit.
    shortcut_strength: float = 0.9


@dataclass
class ModelConfig:
    image_backbone: str = "swin_tiny_patch4_window7_224"
    text_backbone: str = "emilyalsentzer/Bio_ClinicalBERT"
    # Falls back automatically to this if the ClinicalBERT download fails
    # (e.g. offline) so the pipeline never hard-crashes.
    text_backbone_fallback: str = "distilbert-base-uncased"

    fusion_dim: int = 256
    fusion_heads: int = 4
    fusion_layers: int = 2
    dropout: float = 0.3              # also used as MC-Dropout rate at inference
    adapter_dim: int = 64            # bottleneck width of per-domain adapters

    pretrained_image: bool = True
    freeze_text_layers: int = 8       # freeze lower BERT layers to save memory


@dataclass
class TrainConfig:
    epochs_per_domain: int = 4        # # scale-up (8-15 for full results)
    batch_size: int = 8              # T4-safe with Swin-Tiny + ClinicalBERT
    grad_accum_steps: int = 2         # effective batch = 16
    lr: float = 2e-4
    text_lr: float = 1e-5             # smaller LR for the pretrained text encoder
    weight_decay: float = 1e-4
    amp: bool = True                  # mixed precision -> ~2x less VRAM
    max_grad_norm: float = 1.0

    # Continual-learning hyper-parameters
    use_ewc: bool = True
    ewc_lambda: float = 50.0
    use_replay: bool = True
    replay_size: int = 200            # exemplars retained per past domain
    replay_ratio: float = 0.5         # fraction of a batch drawn from replay
    use_adapters: bool = True

    # Causal hyper-parameters
    use_irm: bool = True
    irm_lambda: float = 1.0
    irm_anneal_epochs: int = 1        # warm-up before applying full IRM penalty
    use_causal_reg: bool = True
    causal_reg_lambda: float = 0.1


@dataclass
class Config:
    seed: int = 42
    domains: List[str] = field(default_factory=lambda: ["CheXpert", "MIMIC-CXR", "VinDr-CXR"])
    labels: List[str] = field(default_factory=lambda: list(LABELS))
    output_dir: str = "results"
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

    data: DataConfig = field(default_factory=DataConfig)
    model: ModelConfig = field(default_factory=ModelConfig)
    train: TrainConfig = field(default_factory=TrainConfig)

    @property
    def num_classes(self) -> int:
        return len(self.labels)

    def to_dict(self) -> dict:
        return asdict(self)

    def save(self, path: str) -> None:
        with open(path, "w") as f:
            json.dump(self.to_dict(), f, indent=2)

    @classmethod
    def fast_debug(cls) -> "Config":
        """Tiny config for smoke-testing the whole pipeline in <1 min."""
        c = cls()
        c.data.train_per_domain = 40
        c.data.val_per_domain = 16
        c.data.test_per_domain = 24
        c.train.epochs_per_domain = 1
        c.train.batch_size = 4
        c.train.grad_accum_steps = 1
        c.train.replay_size = 16
        return c


In [ ]:
%%writefile cdime_ai/utils.py
"""Shared utilities: seeding, logging, device helpers, checkpointing."""
from __future__ import annotations

import os
import random
import logging
from typing import Optional

import numpy as np
import torch


def set_seed(seed: int = 42) -> None:
    """Seed every RNG for reproducible (seed-averaged) experiments."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Deterministic cuDNN is slower; keep benchmark on for T4 speed but seed all.
    torch.backends.cudnn.benchmark = True


def get_logger(name: str = "cdime") -> logging.Logger:
    logger = logging.getLogger(name)
    if not logger.handlers:
        handler = logging.StreamHandler()
        handler.setFormatter(logging.Formatter("[%(asctime)s] %(message)s", "%H:%M:%S"))
        logger.addHandler(handler)
        logger.setLevel(logging.INFO)
    return logger


def count_parameters(model: torch.nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def move_batch(batch: dict, device: str) -> dict:
    """Move a dict batch of tensors to ``device`` (non-tensor values untouched)."""
    out = {}
    for k, v in batch.items():
        out[k] = v.to(device, non_blocking=True) if torch.is_tensor(v) else v
    return out


def save_checkpoint(model: torch.nn.Module, path: str) -> None:
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    torch.save(model.state_dict(), path)


def load_checkpoint(model: torch.nn.Module, path: str, device: str = "cpu") -> torch.nn.Module:
    state = torch.load(path, map_location=device)
    model.load_state_dict(state, strict=False)
    return model


In [ ]:
%%writefile cdime_ai/tokenizer.py
"""Tokenizer loader (ClinicalBERT with offline-safe fallback)."""
from __future__ import annotations

from .config import Config


def get_tokenizer(cfg: Config):
    from transformers import AutoTokenizer
    try:
        return AutoTokenizer.from_pretrained(cfg.model.text_backbone)
    except Exception as e:  # pragma: no cover - offline
        print(f"[tokenizer] {cfg.model.text_backbone} unavailable ({e}); "
              f"using {cfg.model.text_backbone_fallback}.")
        return AutoTokenizer.from_pretrained(cfg.model.text_backbone_fallback)


In [ ]:
%%writefile cdime_ai/data/__init__.py
"""CDIME-AI data subpackage."""


In [ ]:
%%writefile cdime_ai/data/synthetic.py
"""Synthetic multimodal chest-X-ray generator with a *causal* structure.

Why synthetic data?  The real datasets (CheXpert ~440 GB raw, MIMIC-CXR ~377 GB)
cannot be downloaded on free Colab. To make the *entire* CDIME-AI pipeline —
continual learning, causal regularisation, ablations and statistical testing —
runnable and reproducible on a T4, we generate data that mimics the multimodal,
multi-label, domain-shifted structure of the real task.

Causal design (this is the important part for the paper):
    * Each image embeds a *causal* lesion signal whose appearance is stable
      across domains -> P(Y | causal features) is invariant.
    * Each image also embeds a *spurious shortcut* (a corner marker) whose
      correlation with the label **flips across domains**. An ERM model latches
      onto the shortcut and fails on the next domain; an IRM / causally
      regularised model learns the invariant signal and generalises.

This lets us empirically demonstrate the value of the causal module exactly as
claimed in the proposal's Research Gap (shortcut learning) and Ablation A1.
"""
from __future__ import annotations

from typing import List, Tuple
import numpy as np
import torch
from torch.utils.data import Dataset

from ..config import Config


# Per-pathology canonical lesion location (row, col) and phrasing for reports.
_LESION_SITES = {
    "Atelectasis":      (0.30, 0.30),
    "Cardiomegaly":     (0.60, 0.50),
    "Consolidation":    (0.40, 0.70),
    "Edema":            (0.50, 0.40),
    "Pleural Effusion": (0.80, 0.65),
}
_PHRASES = {
    "Atelectasis":      "patchy volume loss in the left upper zone",
    "Cardiomegaly":     "enlarged cardiac silhouette",
    "Consolidation":    "dense airspace opacity in the right lung",
    "Edema":            "bilateral perihilar haziness",
    "Pleural Effusion": "blunting of the right costophrenic angle",
}
_NEGATION = "no acute cardiopulmonary abnormality"


def _draw_blob(img: np.ndarray, cy: float, cx: float, intensity: float, sigma: float) -> None:
    """Add a soft Gaussian blob (a lesion) to a single-channel image in place."""
    h, w = img.shape
    yy, xx = np.mgrid[0:h, 0:w]
    y0, x0 = cy * h, cx * w
    blob = intensity * np.exp(-(((yy - y0) ** 2 + (xx - x0) ** 2) / (2 * (sigma * h) ** 2)))
    img += blob


class SyntheticCXRDataset(Dataset):
    """Generates (image, tokenized report, multi-label) triples for one domain.

    Parameters
    ----------
    domain_idx : index of the domain in the continual sequence; controls the
        direction of the spurious shortcut so it flips between domains.
    """

    def __init__(
        self,
        cfg: Config,
        tokenizer,
        n: int,
        domain_idx: int,
        split: str = "train",
        seed: int = 0,
    ):
        self.cfg = cfg
        self.labels = cfg.labels
        self.n_classes = len(self.labels)
        self.size = cfg.data.image_size
        self.tokenizer = tokenizer
        self.max_len = cfg.data.max_text_len
        self.domain_idx = domain_idx
        self.shortcut_strength = cfg.data.shortcut_strength

        rng = np.random.default_rng(seed + 1000 * domain_idx + (0 if split == "train" else 7))
        # Multi-label targets (each pathology present independently ~30%).
        self.y = (rng.random((n, self.n_classes)) < 0.3).astype(np.float32)
        # Ensure no all-zero degenerate rows dominate.
        empties = self.y.sum(1) == 0
        self.y[empties, rng.integers(0, self.n_classes, empties.sum())] = 1.0
        self.seeds = rng.integers(0, 2 ** 31 - 1, size=n)
        # Domain-specific global intensity / contrast shift (covariate shift).
        self.domain_bias = 0.05 * domain_idx
        self.domain_contrast = 1.0 + 0.1 * domain_idx

    def __len__(self) -> int:
        return len(self.y)

    def _render(self, idx: int) -> np.ndarray:
        rng = np.random.default_rng(self.seeds[idx])
        s = self.size
        # Base "lung field" texture.
        img = 0.25 + 0.05 * rng.standard_normal((s, s)).astype(np.float32)
        img = np.clip(img, 0, 1)
        y = self.y[idx]

        # --- Causal lesion signals (domain-invariant) ---
        for c, present in enumerate(y):
            if present:
                cy, cx = _LESION_SITES[self.labels[c]]
                _draw_blob(img, cy, cx, intensity=0.6, sigma=0.06)

        # --- Spurious shortcut (domain-dependent direction) ---
        # A small bright corner marker. In even domains it correlates with the
        # FIRST positive label; in odd domains the correlation is inverted.
        first_pos = int(y.argmax()) if y.sum() > 0 else 0
        corr = self.shortcut_strength
        flip = -1 if (self.domain_idx % 2 == 1) else 1
        # Marker present iff (label parity) agrees with domain-flipped rule.
        rule = (first_pos % 2 == 0)
        marker = (rng.random() < corr) == (rule if flip == 1 else not rule)
        if marker:
            img[:14, :14] += 0.7  # top-left corner shortcut patch

        # --- Domain covariate shift ---
        img = np.clip((img + self.domain_bias) * self.domain_contrast, 0, 1)
        return img.astype(np.float32)

    def _report(self, idx: int) -> str:
        y = self.y[idx]
        findings = [_PHRASES[self.labels[c]] for c, p in enumerate(y) if p]
        if not findings:
            return _NEGATION + "."
        return "Findings: " + "; ".join(findings) + "."

    def __getitem__(self, idx: int):
        img = self._render(idx)                       # (H, W) in [0,1]
        img = torch.from_numpy(img).unsqueeze(0).repeat(3, 1, 1)  # 3-channel
        # ImageNet-style normalisation (matches timm Swin pretraining).
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        img = (img - mean) / std

        report = self._report(idx)
        enc = self.tokenizer(
            report,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt",
        )
        return {
            "image": img,
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label": torch.from_numpy(self.y[idx]),
            "domain": torch.tensor(self.domain_idx, dtype=torch.long),
        }


In [ ]:
%%writefile cdime_ai/data/datasets.py
"""Real-dataset adapters for CheXpert, MIMIC-CXR and VinDr-CXR.

These are used when ``cfg.data.mode == "real"``. They expect a small CSV per
domain (path + label columns) so you can point them at *subsets* that fit on
Colab disk. If a CSV/image is missing the loader raises a clear error telling
you what to download.

Expected CSV columns
--------------------
CheXpert / MIMIC-CXR: standard CheXpert label columns (-1 uncertain handled via
the U-Ones policy -> 1). MIMIC additionally needs a ``report`` text column.
VinDr-CXR: a ``findings`` multi-label set; we map to the 5 shared pathologies.

For full-scale runs replace these with the official loaders; the rest of the
framework is dataset-agnostic (it only consumes the batch dict produced here).
"""
from __future__ import annotations

import os
from typing import Optional

import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset

from ..config import Config


def _u_ones(v: float) -> float:
    """CheXpert uncertainty policy: map uncertain (-1) and NaN to positive/neg."""
    if v == 1:
        return 1.0
    if v == -1:       # U-Ones: treat uncertain as positive
        return 1.0
    return 0.0


class RealCXRDataset(Dataset):
    def __init__(self, cfg: Config, tokenizer, csv_path: str, img_root: str,
                 domain_idx: int, split: str = "train", limit: Optional[int] = None):
        if not os.path.exists(csv_path):
            raise FileNotFoundError(
                f"CSV not found: {csv_path}. Download the dataset subset and "
                f"point cfg at it. See cdime_ai/data/datasets.py docstring."
            )
        self.cfg = cfg
        self.tokenizer = tokenizer
        self.img_root = img_root
        self.labels = cfg.labels
        self.size = cfg.data.image_size
        self.max_len = cfg.data.max_text_len
        self.domain_idx = domain_idx

        df = pd.read_csv(csv_path)
        if limit:
            df = df.sample(n=min(limit, len(df)), random_state=cfg.seed).reset_index(drop=True)
        self.df = df
        self.path_col = "Path" if "Path" in df.columns else "path"
        self.text_col = "report" if "report" in df.columns else None

    def __len__(self) -> int:
        return len(self.df)

    def _load_image(self, rel_path: str) -> torch.Tensor:
        path = os.path.join(self.img_root, rel_path)
        if not os.path.exists(path):
            raise FileNotFoundError(f"Image missing: {path}")
        img = Image.open(path).convert("RGB").resize((self.size, self.size))
        arr = np.asarray(img, dtype=np.float32) / 255.0
        t = torch.from_numpy(arr).permute(2, 0, 1)
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        return (t - mean) / std

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        img = self._load_image(str(row[self.path_col]))
        y = np.array([_u_ones(row.get(l, 0)) for l in self.labels], dtype=np.float32)

        text = str(row[self.text_col]) if self.text_col else "chest radiograph"
        enc = self.tokenizer(
            text, padding="max_length", truncation=True,
            max_length=self.max_len, return_tensors="pt",
        )
        return {
            "image": img,
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label": torch.from_numpy(y),
            "domain": torch.tensor(self.domain_idx, dtype=torch.long),
        }


# Map domain name -> (csv path, image root). Edit these for your Colab layout.
REAL_PATHS = {
    "CheXpert":  ("data/chexpert/subset.csv",  "data/chexpert"),
    "MIMIC-CXR": ("data/mimic/subset.csv",     "data/mimic"),
    "VinDr-CXR": ("data/vindr/subset.csv",     "data/vindr"),
}


In [ ]:
%%writefile cdime_ai/data/loaders.py
"""Build per-domain train/val/test DataLoaders for the continual sequence."""
from __future__ import annotations

from typing import Dict
from torch.utils.data import DataLoader

from ..config import Config
from .synthetic import SyntheticCXRDataset
from .datasets import RealCXRDataset, REAL_PATHS


def _make_loader(ds, cfg: Config, shuffle: bool) -> DataLoader:
    return DataLoader(
        ds,
        batch_size=cfg.train.batch_size,
        shuffle=shuffle,
        num_workers=cfg.data.num_workers,
        pin_memory=cfg.data.pin_memory,
        drop_last=shuffle,
    )


def build_domain_loaders(cfg: Config, tokenizer) -> Dict[str, Dict[str, DataLoader]]:
    """Return ``{domain_name: {"train"/"val"/"test": DataLoader}}``."""
    out: Dict[str, Dict[str, DataLoader]] = {}
    for di, domain in enumerate(cfg.domains):
        loaders = {}
        if cfg.data.mode == "synthetic":
            sizes = {
                "train": cfg.data.train_per_domain,
                "val": cfg.data.val_per_domain,
                "test": cfg.data.test_per_domain,
            }
            for split, n in sizes.items():
                ds = SyntheticCXRDataset(cfg, tokenizer, n=n, domain_idx=di,
                                         split=split, seed=cfg.seed)
                loaders[split] = _make_loader(ds, cfg, shuffle=(split == "train"))
        else:
            csv_path, img_root = REAL_PATHS[domain]
            limits = {
                "train": cfg.data.train_per_domain,
                "val": cfg.data.val_per_domain,
                "test": cfg.data.test_per_domain,
            }
            for split, lim in limits.items():
                ds = RealCXRDataset(cfg, tokenizer, csv_path, img_root,
                                    domain_idx=di, split=split, limit=lim)
                loaders[split] = _make_loader(ds, cfg, shuffle=(split == "train"))
        out[domain] = loaders
    return out


In [ ]:
%%writefile cdime_ai/models/__init__.py
"""CDIME-AI models subpackage."""


In [ ]:
%%writefile cdime_ai/models/image_encoder.py
"""Swin Transformer image encoder (timm) exposing spatial tokens for Grad-CAM.

Returns both a sequence of patch tokens (for cross-attention fusion and
attention-rollout explainability) and the final spatial feature map (for
Grad-CAM). Falls back to a small ResNet if timm/Swin is unavailable so the
pipeline still runs.
"""
from __future__ import annotations

import torch
import torch.nn as nn


class SwinImageEncoder(nn.Module):
    def __init__(self, backbone: str = "swin_tiny_patch4_window7_224",
                 pretrained: bool = True, out_dim: int = 256):
        super().__init__()
        self.out_dim = out_dim
        self._kind = None
        try:
            import timm
            # features_only gives us the last spatial map (H', W', C) for Grad-CAM.
            self.backbone = timm.create_model(
                backbone, pretrained=pretrained, features_only=True, out_indices=(3,)
            )
            self.feat_dim = self.backbone.feature_info.channels()[-1]
            self._kind = "swin"
        except Exception as e:  # pragma: no cover - offline / no timm
            print(f"[image_encoder] timm Swin unavailable ({e}); using ResNet fallback.")
            from torchvision.models import resnet18, ResNet18_Weights
            w = ResNet18_Weights.DEFAULT if pretrained else None
            net = resnet18(weights=w)
            self.backbone = nn.Sequential(*list(net.children())[:-2])  # -> (B,512,h,w)
            self.feat_dim = 512
            self._kind = "resnet"

        self.proj = nn.Linear(self.feat_dim, out_dim)
        # Cache the last spatial feature map for Grad-CAM hooks.
        self.last_feature_map: torch.Tensor | None = None

    def forward(self, x: torch.Tensor):
        """Return (tokens, pooled).

        tokens : (B, N, out_dim) patch-token sequence for fusion / attention maps
        pooled : (B, out_dim)    mean-pooled global image embedding
        """
        if self._kind == "swin":
            feat = self.backbone(x)[-1]            # timm Swin: (B, H, W, C)
            if feat.dim() == 4 and feat.shape[-1] == self.feat_dim:
                # channels-last (H,W,C) -> (B,C,H,W)
                feat = feat.permute(0, 3, 1, 2).contiguous()
        else:
            feat = self.backbone(x)                # (B, C, H, W)

        feat.retain_grad() if feat.requires_grad else None
        self.last_feature_map = feat               # (B, C, H, W) for Grad-CAM

        b, c, h, w = feat.shape
        tokens = feat.flatten(2).transpose(1, 2)   # (B, N=h*w, C)
        tokens = self.proj(tokens)                 # (B, N, out_dim)
        pooled = tokens.mean(dim=1)                # (B, out_dim)
        return tokens, pooled


In [ ]:
%%writefile cdime_ai/models/text_encoder.py
"""ClinicalBERT text encoder for radiology reports (with DistilBERT fallback)."""
from __future__ import annotations

import torch
import torch.nn as nn


class ClinicalBERTEncoder(nn.Module):
    def __init__(self, backbone: str = "emilyalsentzer/Bio_ClinicalBERT",
                 fallback: str = "distilbert-base-uncased",
                 out_dim: int = 256, freeze_layers: int = 8):
        super().__init__()
        from transformers import AutoModel
        try:
            self.bert = AutoModel.from_pretrained(backbone)
            self.name = backbone
        except Exception as e:  # pragma: no cover - offline
            print(f"[text_encoder] {backbone} unavailable ({e}); falling back to {fallback}.")
            self.bert = AutoModel.from_pretrained(fallback)
            self.name = fallback

        hidden = self.bert.config.hidden_size
        self.proj = nn.Linear(hidden, out_dim)
        self.out_dim = out_dim
        self._freeze_lower(freeze_layers)

    def _freeze_lower(self, n: int) -> None:
        """Freeze embeddings + the lowest ``n`` transformer layers to save memory."""
        if n <= 0:
            return
        for p in self.bert.embeddings.parameters():
            p.requires_grad = False
        layers = None
        if hasattr(self.bert, "encoder") and hasattr(self.bert.encoder, "layer"):
            layers = self.bert.encoder.layer            # BERT-style
        elif hasattr(self.bert, "transformer") and hasattr(self.bert.transformer, "layer"):
            layers = self.bert.transformer.layer        # DistilBERT-style
        if layers is not None:
            for layer in layers[:n]:
                for p in layer.parameters():
                    p.requires_grad = False

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor):
        """Return (tokens, pooled).

        tokens : (B, L, out_dim) projected token embeddings for cross-attention
        pooled : (B, out_dim)    CLS-token (or mean) embedding
        """
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        seq = out.last_hidden_state                    # (B, L, hidden)
        tokens = self.proj(seq)                         # (B, L, out_dim)
        # Masked mean pooling as the global text embedding.
        mask = attention_mask.unsqueeze(-1).float()
        pooled = (tokens * mask).sum(1) / mask.sum(1).clamp(min=1e-6)
        return tokens, pooled


In [ ]:
%%writefile cdime_ai/models/adapters.py
"""Per-domain bottleneck adapters for parameter-efficient continual learning.

Each domain gets its own lightweight residual adapter (Houlsby-style). Only the
active domain's adapter is trained while the shared backbone is regularised by
EWC, giving plasticity for the new domain without overwriting shared features.
"""
from __future__ import annotations

import torch
import torch.nn as nn


class Adapter(nn.Module):
    def __init__(self, dim: int, bottleneck: int = 64, dropout: float = 0.1):
        super().__init__()
        self.down = nn.Linear(dim, bottleneck)
        self.act = nn.GELU()
        self.up = nn.Linear(bottleneck, dim)
        self.drop = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(dim)
        nn.init.zeros_(self.up.weight)   # start as identity (residual)
        nn.init.zeros_(self.up.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.up(self.drop(self.act(self.down(x))))
        return self.norm(x + h)


class DomainAdapterBank(nn.Module):
    """A bank of adapters, one per domain, selected by ``active_domain``."""

    def __init__(self, dim: int, n_domains: int, bottleneck: int = 64, dropout: float = 0.1):
        super().__init__()
        self.adapters = nn.ModuleList(
            [Adapter(dim, bottleneck, dropout) for _ in range(n_domains)]
        )
        self.active_domain = 0

    def set_domain(self, idx: int) -> None:
        self.active_domain = idx

    def forward(self, x: torch.Tensor, domain: int | None = None) -> torch.Tensor:
        idx = self.active_domain if domain is None else domain
        return self.adapters[idx](x)


In [ ]:
%%writefile cdime_ai/models/fusion.py
"""Cross-attention transformer fusion of image patch tokens and report tokens.

Image tokens attend to text tokens and vice-versa over several layers; the
fused [CLS]-style summary tokens are concatenated into the joint representation.
Attention weights from the last layer are cached for the attention-map
explainability module.
"""
from __future__ import annotations

import torch
import torch.nn as nn


class CrossAttentionBlock(nn.Module):
    def __init__(self, dim: int, heads: int, dropout: float):
        super().__init__()
        self.attn = nn.MultiheadAttention(dim, heads, dropout=dropout, batch_first=True)
        self.norm_q = nn.LayerNorm(dim)
        self.norm_kv = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(
            nn.Linear(dim, dim * 2), nn.GELU(), nn.Dropout(dropout), nn.Linear(dim * 2, dim)
        )
        self.norm_ffn = nn.LayerNorm(dim)
        self.last_attn: torch.Tensor | None = None

    def forward(self, q, kv, kv_mask=None):
        qn, kvn = self.norm_q(q), self.norm_kv(kv)
        key_padding = (kv_mask == 0) if kv_mask is not None else None
        out, attn = self.attn(qn, kvn, kvn, key_padding_mask=key_padding,
                              need_weights=True, average_attn_weights=True)
        self.last_attn = attn.detach()                 # (B, Lq, Lkv)
        q = q + out
        q = q + self.ffn(self.norm_ffn(q))
        return q


class CrossAttentionFusion(nn.Module):
    def __init__(self, dim: int = 256, heads: int = 4, layers: int = 2, dropout: float = 0.3):
        super().__init__()
        self.img2txt = nn.ModuleList(
            [CrossAttentionBlock(dim, heads, dropout) for _ in range(layers)])
        self.txt2img = nn.ModuleList(
            [CrossAttentionBlock(dim, heads, dropout) for _ in range(layers)])
        self.out_dim = dim * 2

    def forward(self, img_tokens, txt_tokens, txt_mask=None):
        i, t = img_tokens, txt_tokens
        for blk_it, blk_ti in zip(self.img2txt, self.txt2img):
            new_i = blk_it(i, t, txt_mask)             # image attends to text
            new_t = blk_ti(t, i, None)                 # text attends to image
            i, t = new_i, new_t
        img_summary = i.mean(dim=1)                     # (B, dim)
        txt_summary = t.mean(dim=1)                     # (B, dim)
        fused = torch.cat([img_summary, txt_summary], dim=-1)   # (B, 2*dim)
        return fused

    @property
    def last_cross_attention(self) -> torch.Tensor | None:
        """Image->text attention from the final fusion layer (B, N_img, L_txt)."""
        return self.img2txt[-1].last_attn


In [ ]:
%%writefile cdime_ai/models/cdime_model.py
"""CDIME-AI: the full multimodal causal continual model.

Combines the Swin image encoder, ClinicalBERT text encoder, cross-attention
fusion, per-domain adapters and a multi-label classifier head. Ablation
switches let a single class realise every configuration in proposal Section 7.
"""
from __future__ import annotations

from dataclasses import dataclass
from typing import Optional

import torch
import torch.nn as nn

from ..config import Config
from .image_encoder import SwinImageEncoder
from .text_encoder import ClinicalBERTEncoder
from .fusion import CrossAttentionFusion
from .adapters import DomainAdapterBank


@dataclass
class AblationFlags:
    use_text: bool = True          # A5: False -> image-only
    use_cross_attention: bool = True  # A6: False -> simple concatenation
    use_adapters: bool = True      # A2/A7
    enable_mc_dropout: bool = True # A4 (uncertainty handled in trainer)


class ConcatFusion(nn.Module):
    """Simple concatenation baseline for ablation A6."""

    def __init__(self, dim: int):
        super().__init__()
        self.out_dim = dim * 2
        self.last_cross_attention = None

    def forward(self, img_tokens, txt_tokens, txt_mask=None):
        return torch.cat([img_tokens.mean(1), txt_tokens.mean(1)], dim=-1)


class CDIMEModel(nn.Module):
    def __init__(self, cfg: Config, ablation: Optional[AblationFlags] = None):
        super().__init__()
        self.cfg = cfg
        self.ab = ablation or AblationFlags()
        d = cfg.model.fusion_dim
        n_domains = len(cfg.domains)

        self.image_encoder = SwinImageEncoder(
            cfg.model.image_backbone, cfg.model.pretrained_image, out_dim=d)

        if self.ab.use_text:
            self.text_encoder = ClinicalBERTEncoder(
                cfg.model.text_backbone, cfg.model.text_backbone_fallback,
                out_dim=d, freeze_layers=cfg.model.freeze_text_layers)
            if self.ab.use_cross_attention:
                self.fusion = CrossAttentionFusion(
                    d, cfg.model.fusion_heads, cfg.model.fusion_layers, cfg.model.dropout)
            else:
                self.fusion = ConcatFusion(d)
            fused_dim = self.fusion.out_dim
        else:
            self.text_encoder = None
            self.fusion = None
            fused_dim = d

        self.use_adapters = self.ab.use_adapters
        if self.use_adapters:
            self.adapters = DomainAdapterBank(
                fused_dim, n_domains, cfg.model.adapter_dim, cfg.model.dropout)

        self.dropout = nn.Dropout(cfg.model.dropout)   # MC-Dropout layer
        self.classifier = nn.Sequential(
            nn.Linear(fused_dim, fused_dim // 2),
            nn.GELU(),
            nn.Dropout(cfg.model.dropout),
            nn.Linear(fused_dim // 2, cfg.num_classes),
        )

    def set_domain(self, idx: int) -> None:
        if self.use_adapters:
            self.adapters.set_domain(idx)

    def enable_mc_dropout(self) -> None:
        """Put ONLY dropout layers in train mode for MC-Dropout sampling."""
        self.eval()
        for m in self.modules():
            if isinstance(m, nn.Dropout):
                m.train()

    def encode(self, batch, domain: Optional[int] = None) -> torch.Tensor:
        img_tokens, img_pooled = self.image_encoder(batch["image"])
        if self.ab.use_text and self.text_encoder is not None:
            txt_tokens, _ = self.text_encoder(batch["input_ids"], batch["attention_mask"])
            fused = self.fusion(img_tokens, txt_tokens, batch["attention_mask"])
        else:
            fused = img_pooled
        if self.use_adapters:
            fused = self.adapters(fused, domain)
        return fused

    def forward(self, batch, domain: Optional[int] = None,
                return_features: bool = False):
        features = self.encode(batch, domain)
        logits = self.classifier(self.dropout(features))
        if return_features:
            return logits, features
        return logits


In [ ]:
%%writefile cdime_ai/causal/__init__.py
"""CDIME-AI causal subpackage."""


In [ ]:
%%writefile cdime_ai/causal/irm.py
"""Causal learning objectives: IRM + V-REx (causal regularisation).

Conceptual framing (proposal Section 5, "Causal Module: SCM, IRM, causal
regularization"):

    We treat each hospital/domain as an *environment* drawn from a Structural
    Causal Model in which the disease -> imaging-finding mechanism is invariant,
    while nuisance/shortcut variables (scanner, markers, demographics) vary.
    Invariant Risk Minimisation (IRM) seeks a representation whose optimal
    classifier is simultaneously optimal in every environment, i.e. it relies
    only on the stable causal mechanism. V-REx adds an explicit penalty on the
    variance of risk across environments (causal regularisation), discouraging
    the model from exploiting environment-specific shortcuts.
"""
from __future__ import annotations

from typing import Dict
import torch
import torch.nn.functional as F


def _bce(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    return F.binary_cross_entropy_with_logits(logits, targets)


def irm_penalty(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    """IRMv1 gradient penalty for one environment.

    Multiplies the logits by a constant dummy classifier ``w = 1`` and returns
    the squared gradient of the risk w.r.t. ``w``. A small penalty means the
    representation's classifier is already locally optimal in this environment.
    """
    scale = torch.ones(1, device=logits.device, requires_grad=True)
    loss = _bce(logits * scale, targets)
    grad = torch.autograd.grad(loss, [scale], create_graph=True)[0]
    return (grad ** 2).sum()


def environment_losses(logits: torch.Tensor, targets: torch.Tensor,
                       domains: torch.Tensor) -> Dict[int, torch.Tensor]:
    """Split a mixed-domain batch into per-environment BCE losses."""
    losses: Dict[int, torch.Tensor] = {}
    for d in domains.unique():
        m = domains == d
        if m.sum() > 0:
            losses[int(d.item())] = _bce(logits[m], targets[m])
    return losses


def causal_objective(logits: torch.Tensor, targets: torch.Tensor,
                     domains: torch.Tensor, irm_lambda: float = 1.0,
                     vrex_lambda: float = 0.1):
    """Return (irm_penalty, vrex_penalty) summed/averaged over environments.

    When a batch contains a single environment (common in the continual setting
    where each step trains on one domain), the replay buffer supplies exemplars
    from past domains so multiple environments co-occur and these penalties
    become meaningful. With one environment, V-REx is 0 and IRM still pushes
    toward a locally-invariant classifier.
    """
    env_logits, env_targets = {}, {}
    for d in domains.unique():
        m = domains == d
        env_logits[int(d)] = logits[m]
        env_targets[int(d)] = targets[m]

    irm_terms, risks = [], []
    for d in env_logits:
        if env_logits[d].shape[0] == 0:
            continue
        irm_terms.append(irm_penalty(env_logits[d], env_targets[d]))
        risks.append(_bce(env_logits[d], env_targets[d]))

    irm_val = torch.stack(irm_terms).mean() if irm_terms else logits.sum() * 0.0
    if len(risks) > 1:
        vrex_val = torch.stack(risks).var(unbiased=False)
    else:
        vrex_val = logits.sum() * 0.0
    return irm_lambda * irm_val, vrex_lambda * vrex_val


In [ ]:
%%writefile cdime_ai/continual/__init__.py
"""CDIME-AI continual subpackage."""


In [ ]:
%%writefile cdime_ai/continual/ewc.py
"""Elastic Weight Consolidation (EWC) for catastrophic-forgetting mitigation.

After finishing a domain we estimate the diagonal Fisher information of the
shared parameters and anchor them. While training the next domain a quadratic
penalty keeps important weights near their consolidated values.

Supports multiple anchored tasks (online sum of per-task penalties).
"""
from __future__ import annotations

from typing import Dict, List
import torch
import torch.nn as nn
import torch.nn.functional as F

from ..utils import move_batch


class EWC:
    def __init__(self, ewc_lambda: float = 50.0):
        self.ewc_lambda = ewc_lambda
        self.tasks: List[Dict[str, torch.Tensor]] = []   # stored {param_name: theta*}
        self.fishers: List[Dict[str, torch.Tensor]] = []  # stored {param_name: F}

    @staticmethod
    def _trainable_named_params(model: nn.Module):
        return {n: p for n, p in model.named_parameters() if p.requires_grad}

    @torch.enable_grad()
    def consolidate(self, model: nn.Module, loader, device: str,
                    domain_idx: int, max_batches: int = 30) -> None:
        """Estimate diagonal Fisher on ``loader`` and snapshot current weights."""
        model.eval()
        params = self._trainable_named_params(model)
        fisher = {n: torch.zeros_like(p) for n, p in params.items()}

        n_seen = 0
        for i, batch in enumerate(loader):
            if i >= max_batches:
                break
            batch = move_batch(batch, device)
            model.zero_grad()
            logits = model(batch, domain=domain_idx)
            # Sample-style Fisher: gradient of the log-likelihood of the
            # model's own predictions (Bernoulli per label).
            probs = torch.sigmoid(logits).detach()
            loss = F.binary_cross_entropy_with_logits(logits, probs)
            loss.backward()
            for n, p in params.items():
                if p.grad is not None:
                    fisher[n] += p.grad.detach() ** 2 * batch["image"].size(0)
            n_seen += batch["image"].size(0)

        for n in fisher:
            fisher[n] /= max(n_seen, 1)

        self.fishers.append(fisher)
        self.tasks.append({n: p.detach().clone() for n, p in params.items()})
        model.zero_grad()

    def penalty(self, model: nn.Module) -> torch.Tensor:
        """Quadratic EWC penalty summed over all consolidated tasks."""
        if not self.tasks:
            return torch.zeros((), device=next(model.parameters()).device)
        params = self._trainable_named_params(model)
        total = torch.zeros((), device=next(model.parameters()).device)
        for theta_star, fisher in zip(self.tasks, self.fishers):
            for n, p in params.items():
                if n in theta_star and n in fisher:
                    total = total + (fisher[n] * (p - theta_star[n]) ** 2).sum()
        return 0.5 * self.ewc_lambda * total


In [ ]:
%%writefile cdime_ai/continual/replay.py
"""Replay memory (experience rehearsal) for domain-incremental learning.

Stores a bounded set of exemplars (full multimodal samples) from each past
domain using reservoir sampling. During training on a new domain, a fraction of
each mini-batch is drawn from replay so that (a) past domains are rehearsed and
(b) batches contain *multiple environments*, which is what makes the IRM/V-REx
causal penalties effective.
"""
from __future__ import annotations

from typing import Dict, List, Optional
import random
import torch


class ReplayMemory:
    def __init__(self, capacity_per_domain: int = 200):
        self.capacity = capacity_per_domain
        self.buffer: Dict[int, List[dict]] = {}
        self._count: Dict[int, int] = {}

    def add_loader(self, loader, domain_idx: int) -> None:
        """Reservoir-sample exemplars from a finished domain's train loader."""
        self.buffer.setdefault(domain_idx, [])
        self._count.setdefault(domain_idx, 0)
        buf = self.buffer[domain_idx]
        for batch in loader:
            b = batch["image"].size(0)
            for i in range(b):
                sample = {k: (v[i].detach().cpu() if torch.is_tensor(v) else v)
                          for k, v in batch.items()}
                self._count[domain_idx] += 1
                if len(buf) < self.capacity:
                    buf.append(sample)
                else:
                    j = random.randint(0, self._count[domain_idx] - 1)
                    if j < self.capacity:
                        buf[j] = sample

    def is_empty(self) -> bool:
        return sum(len(v) for v in self.buffer.values()) == 0

    def sample(self, n: int) -> Optional[dict]:
        """Return a collated batch of ``n`` exemplars drawn across past domains."""
        pool = [s for samples in self.buffer.values() for s in samples]
        if not pool or n <= 0:
            return None
        chosen = random.choices(pool, k=min(n, len(pool)))
        keys = chosen[0].keys()
        out = {}
        for k in keys:
            if torch.is_tensor(chosen[0][k]):
                out[k] = torch.stack([c[k] for c in chosen])
            else:
                out[k] = [c[k] for c in chosen]
        return out


In [ ]:
%%writefile cdime_ai/uncertainty/__init__.py
"""CDIME-AI uncertainty subpackage."""


In [ ]:
%%writefile cdime_ai/uncertainty/mc_dropout.py
"""Monte-Carlo Dropout for epistemic uncertainty estimation."""
from __future__ import annotations

import torch


@torch.no_grad()
def mc_dropout_predict(model, batch, domain: int, n_samples: int = 20):
    """Run ``n_samples`` stochastic forward passes with dropout enabled.

    Returns
    -------
    mean_prob : (B, C) predictive mean probability
    std_prob  : (B, C) predictive std (epistemic uncertainty per label)
    entropy   : (B,)   mean predictive entropy across labels
    """
    model.enable_mc_dropout()
    probs = []
    for _ in range(n_samples):
        logits = model(batch, domain=domain)
        probs.append(torch.sigmoid(logits))
    probs = torch.stack(probs, dim=0)          # (S, B, C)
    mean_prob = probs.mean(0)
    std_prob = probs.std(0)
    eps = 1e-8
    ent = -(mean_prob * (mean_prob + eps).log()
            + (1 - mean_prob) * (1 - mean_prob + eps).log())
    return mean_prob, std_prob, ent.mean(dim=1)


In [ ]:
%%writefile cdime_ai/uncertainty/temperature_scaling.py
"""Temperature scaling for post-hoc calibration (Guo et al., 2017).

Learns a single scalar T on a held-out validation set that divides the logits
before the sigmoid, minimising calibration error without changing accuracy.
"""
from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F

from ..utils import move_batch


class TemperatureScaler(nn.Module):
    def __init__(self):
        super().__init__()
        self.log_temp = nn.Parameter(torch.zeros(1))   # T = exp(log_temp), init 1

    @property
    def temperature(self) -> float:
        return float(self.log_temp.exp().item())

    def forward(self, logits: torch.Tensor) -> torch.Tensor:
        return logits / self.log_temp.exp()

    @torch.enable_grad()
    def fit(self, model, loader, domain: int, device: str, max_iter: int = 100):
        """Optimise T on ``loader`` via LBFGS over BCE of scaled logits."""
        model.eval()
        all_logits, all_targets = [], []
        with torch.no_grad():
            for batch in loader:
                batch = move_batch(batch, device)
                all_logits.append(model(batch, domain=domain).detach())
                all_targets.append(batch["label"])
        logits = torch.cat(all_logits)
        targets = torch.cat(all_targets)

        self.to(device)
        opt = torch.optim.LBFGS([self.log_temp], lr=0.05, max_iter=max_iter)

        def closure():
            opt.zero_grad()
            loss = F.binary_cross_entropy_with_logits(self(logits), targets)
            loss.backward()
            return loss

        opt.step(closure)
        return self.temperature


In [ ]:
%%writefile cdime_ai/explain/__init__.py
"""CDIME-AI explain subpackage."""


In [ ]:
%%writefile cdime_ai/explain/gradcam.py
"""Grad-CAM saliency for the image encoder.

Hooks the cached last spatial feature map of the Swin/ResNet backbone and
produces a per-class heat-map indicating where the model looks. Useful both for
clinician trust and for the insertion/deletion faithfulness metric.
"""
from __future__ import annotations

import torch
import torch.nn.functional as F


class GradCAM:
    def __init__(self, model):
        self.model = model
        self._fmap = None
        self._grad = None
        enc = model.image_encoder

        # Forward hook captures the feature map; backward hook captures grads.
        def fwd_hook(_m, _inp, _out):
            self._fmap = enc.last_feature_map
            if self._fmap is not None and self._fmap.requires_grad:
                self._fmap.register_hook(self._save_grad)

        enc.register_forward_hook(fwd_hook)

    def _save_grad(self, grad):
        self._grad = grad

    def __call__(self, batch, class_idx: int, domain: int = 0):
        """Return a (B, H, W) normalised Grad-CAM heat-map for ``class_idx``."""
        self.model.eval()
        self.model.zero_grad()
        logits = self.model(batch, domain=domain)
        score = logits[:, class_idx].sum()
        score.backward()

        fmap, grad = self._fmap, self._grad          # (B, C, H, W)
        weights = grad.mean(dim=(2, 3), keepdim=True)  # GAP over spatial dims
        cam = F.relu((weights * fmap).sum(dim=1))      # (B, H, W)
        # Normalise each map to [0, 1].
        cam = cam - cam.amin(dim=(1, 2), keepdim=True)
        cam = cam / (cam.amax(dim=(1, 2), keepdim=True) + 1e-8)
        img_size = batch["image"].shape[-1]
        cam = F.interpolate(cam.unsqueeze(1), size=(img_size, img_size),
                            mode="bilinear", align_corners=False).squeeze(1)
        return cam.detach()


In [ ]:
%%writefile cdime_ai/explain/attention.py
"""Cross-attention map extraction from the fusion module.

Surfaces which report tokens the image attends to (and the spatial attention
over image patches), giving a multimodal rationale for each prediction.
"""
from __future__ import annotations

import torch


@torch.no_grad()
def image_to_text_attention(model, batch, domain: int = 0):
    """Return mean image->text attention (B, L_txt) over report tokens.

    Higher weight => that report token contributed more to the fused image
    representation. Returns None for image-only / concat-fusion ablations.
    """
    model.eval()
    _ = model(batch, domain=domain)
    fusion = getattr(model, "fusion", None)
    if fusion is None or getattr(fusion, "last_cross_attention", None) is None:
        return None
    attn = fusion.last_cross_attention          # (B, N_img, L_txt)
    return attn.mean(dim=1)                      # average over image queries


def top_text_evidence(attn_row: torch.Tensor, input_ids: torch.Tensor,
                      tokenizer, k: int = 5):
    """Return the top-k report tokens by attention weight (decoded strings)."""
    if attn_row is None:
        return []
    vals, idx = attn_row.topk(min(k, attn_row.numel()))
    toks = tokenizer.convert_ids_to_tokens(input_ids[idx].tolist())
    return [(t, float(v)) for t, v in zip(toks, vals)]


In [ ]:
%%writefile cdime_ai/explain/counterfactual.py
"""Counterfactual explanations.

Answers "what minimal change to the image would flip this diagnosis?" by
gradient ascent/descent on the input toward the decision boundary for a target
class, under an L2 proximity constraint. The resulting difference map highlights
the causally-relevant region (it should overlap the lesion, not the shortcut).
"""
from __future__ import annotations

import torch


def generate_counterfactual(model, batch, class_idx: int, domain: int = 0,
                            target_prob: float = 0.5, steps: int = 30,
                            lr: float = 0.02, proximity: float = 1.0):
    """Return (cf_image, delta, orig_prob, cf_prob) for the first sample.

    Flips the prediction for ``class_idx`` toward ``target_prob`` with a small,
    proximity-regularised perturbation.
    """
    model.eval()
    x = batch["image"][:1].clone().detach()
    single = {k: (v[:1].clone() if torch.is_tensor(v) else v) for k, v in batch.items()}

    with torch.no_grad():
        orig_prob = torch.sigmoid(model(single, domain=domain))[0, class_idx].item()
    # Push toward the opposite class.
    target = 0.0 if orig_prob >= 0.5 else 1.0

    delta = torch.zeros_like(x, requires_grad=True)
    opt = torch.optim.Adam([delta], lr=lr)
    target_t = torch.tensor([[target]], device=x.device)

    for _ in range(steps):
        opt.zero_grad()
        single["image"] = x + delta
        logit = model(single, domain=domain)[:, class_idx:class_idx + 1]
        bce = torch.nn.functional.binary_cross_entropy_with_logits(logit, target_t)
        loss = bce + proximity * (delta ** 2).mean()
        loss.backward()
        opt.step()

    with torch.no_grad():
        single["image"] = x + delta
        cf_prob = torch.sigmoid(model(single, domain=domain))[0, class_idx].item()

    return (x + delta).detach(), delta.detach(), orig_prob, cf_prob


In [ ]:
%%writefile cdime_ai/explain/shap_explain.py
"""Feature attribution via occlusion (a dependency-free SHAP surrogate).

Full KernelSHAP is far too slow for high-res images on a T4. We use occlusion
sensitivity — sliding a grey patch over the image and measuring the drop in the
target-class probability — which is a Shapley-style marginal-contribution
estimate over super-pixels and is the standard practical attribution for CXR.

If the ``shap`` package is installed and ``use_shap=True``, a GradientExplainer
is used on the pooled image embedding instead.
"""
from __future__ import annotations

import torch
import torch.nn.functional as F


@torch.no_grad()
def occlusion_attribution(model, batch, class_idx: int, domain: int = 0,
                          patch: int = 28, stride: int = 28):
    """Return a (B, H, W) importance map from occlusion sensitivity."""
    model.eval()
    x = batch["image"]
    b, _, h, w = x.shape
    base = torch.sigmoid(model(batch, domain=domain))[:, class_idx]   # (B,)
    heat = torch.zeros(b, h, w, device=x.device)
    counts = torch.zeros(b, h, w, device=x.device)

    for i in range(0, h - patch + 1, stride):
        for j in range(0, w - patch + 1, stride):
            occ = x.clone()
            occ[:, :, i:i + patch, j:j + patch] = 0.0    # grey-out (post-norm 0)
            single = dict(batch)
            single["image"] = occ
            p = torch.sigmoid(model(single, domain=domain))[:, class_idx]
            drop = (base - p).clamp(min=0)               # importance = prob drop
            heat[:, i:i + patch, j:j + patch] += drop.view(b, 1, 1)
            counts[:, i:i + patch, j:j + patch] += 1

    heat = heat / counts.clamp(min=1)
    heat = heat - heat.amin(dim=(1, 2), keepdim=True)
    heat = heat / (heat.amax(dim=(1, 2), keepdim=True) + 1e-8)
    return heat


def shap_attribution(model, batch, class_idx: int, domain: int = 0):
    """Optional true-SHAP path; falls back to occlusion if shap is missing."""
    try:
        import shap  # noqa: F401
    except Exception:
        return occlusion_attribution(model, batch, class_idx, domain)
    # For brevity we route to occlusion even when shap is present, since image
    # SHAP requires a background dataset; occlusion is the robust default.
    return occlusion_attribution(model, batch, class_idx, domain)


In [ ]:
%%writefile cdime_ai/decision/__init__.py
"""CDIME-AI decision subpackage."""


In [ ]:
%%writefile cdime_ai/decision/therapeutic.py
"""Therapeutic / follow-up decision support.

Combines the calibrated diagnosis, predictive confidence, epistemic uncertainty
(MC-Dropout) and explanatory evidence into a structured, evidence-based
recommendation. This is a *decision-support* layer (proposal Section 10: the
system never makes autonomous diagnoses) — every recommendation is conditioned
on confidence and flags low-certainty cases for human review.

The rules below are guideline-aligned placeholders; they are intentionally
conservative and should be reviewed by clinicians before any real use.
"""
from __future__ import annotations

from dataclasses import dataclass, field
from typing import List, Dict


# Evidence-based follow-up suggestions per pathology (decision-support only).
_RECOMMENDATIONS: Dict[str, str] = {
    "Atelectasis": "Encourage incentive spirometry / chest physiotherapy; "
                   "correlate with clinical status; consider follow-up imaging.",
    "Cardiomegaly": "Recommend echocardiography and BNP; review cardiac history "
                    "and consider cardiology referral.",
    "Consolidation": "Consider sputum culture and empiric antibiotics if "
                     "infection suspected; follow-up radiograph in 4-6 weeks.",
    "Edema": "Assess fluid status; consider diuretics and echocardiography; "
             "monitor renal function and electrolytes.",
    "Pleural Effusion": "Consider diagnostic thoracentesis if clinically "
                        "significant; ultrasound to characterise the effusion.",
}


@dataclass
class DecisionReport:
    diagnoses: List[Dict] = field(default_factory=list)  # per positive label
    overall_confidence: float = 0.0
    needs_review: bool = False
    recommendations: List[str] = field(default_factory=list)
    notes: str = ""

    def to_text(self) -> str:
        lines = ["=== CDIME-AI Decision Support Report ==="]
        if not self.diagnoses:
            lines.append("No pathology exceeds the decision threshold "
                         "(study likely normal). Correlate clinically.")
        for d in self.diagnoses:
            lines.append(
                f"- {d['label']}: p={d['probability']:.2f} "
                f"(±{d['uncertainty']:.2f}), confidence={d['confidence']}")
        lines.append(f"Overall confidence: {self.overall_confidence:.2f}")
        if self.recommendations:
            lines.append("Suggested follow-up:")
            lines += [f"  * {r}" for r in self.recommendations]
        if self.needs_review:
            lines.append("** LOW CERTAINTY — flagged for radiologist review. **")
        if self.notes:
            lines.append(self.notes)
        return "\n".join(lines)


def make_decision(labels: List[str], probs, uncertainties,
                  evidence: List[str] | None = None,
                  threshold: float = 0.5, high_unc: float = 0.15) -> DecisionReport:
    """Build a structured decision report for a single study.

    Parameters
    ----------
    probs : per-label calibrated probabilities (length C)
    uncertainties : per-label MC-Dropout std (length C)
    evidence : optional list of textual/visual evidence snippets
    """
    report = DecisionReport()
    confidences = []
    for i, label in enumerate(labels):
        p = float(probs[i])
        u = float(uncertainties[i]) if uncertainties is not None else 0.0
        if p >= threshold:
            conf = "high" if (p >= 0.7 and u < high_unc) else "moderate" if p >= 0.55 else "low"
            confidences.append(p * (1 - min(u / high_unc, 1.0)))
            report.diagnoses.append({
                "label": label, "probability": p, "uncertainty": u, "confidence": conf,
            })
            report.recommendations.append(_RECOMMENDATIONS.get(label, "Clinical correlation advised."))

    report.overall_confidence = float(sum(confidences) / len(confidences)) if confidences else 0.0
    # Flag for human review if any positive finding is uncertain or borderline.
    report.needs_review = any(
        d["confidence"] != "high" for d in report.diagnoses
    ) or (len(report.diagnoses) == 0)
    if evidence:
        report.notes = "Evidence: " + "; ".join(evidence)
    return report


In [ ]:
%%writefile cdime_ai/evaluation/__init__.py
"""CDIME-AI evaluation subpackage."""


In [ ]:
%%writefile cdime_ai/evaluation/metrics.py
"""Evaluation metrics: diagnostic, continual-learning, calibration, faithfulness.

All diagnostic metrics are computed in a multi-label setting (macro-averaged
over the 5 pathologies). Continual-learning metrics follow Lopez-Paz & Ranzato
(GEM, 2017) using the accuracy matrix R[i, j] = accuracy on domain j after
training on domain i.
"""
from __future__ import annotations

from typing import Dict, List
import numpy as np
import torch

from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score, accuracy_score,
)


# -----------------------------------------------------------------------------
# Diagnostic metrics
# -----------------------------------------------------------------------------
@torch.no_grad()
def collect_predictions(model, loader, domain: int, device: str):
    model.eval()
    from ..utils import move_batch
    probs, targets = [], []
    for batch in loader:
        batch = move_batch(batch, device)
        logits = model(batch, domain=domain)
        probs.append(torch.sigmoid(logits).cpu().numpy())
        targets.append(batch["label"].cpu().numpy())
    return np.concatenate(probs), np.concatenate(targets)


def diagnostic_metrics(probs: np.ndarray, targets: np.ndarray,
                       threshold: float = 0.5) -> Dict[str, float]:
    preds = (probs >= threshold).astype(int)
    out: Dict[str, float] = {}
    out["accuracy"] = float(accuracy_score(targets.flatten(), preds.flatten()))
    out["precision"] = float(precision_score(targets, preds, average="macro", zero_division=0))
    out["recall"] = float(recall_score(targets, preds, average="macro", zero_division=0))
    out["f1"] = float(f1_score(targets, preds, average="macro", zero_division=0))
    # Specificity (macro): TN / (TN + FP)
    specs = []
    for c in range(targets.shape[1]):
        tn = ((preds[:, c] == 0) & (targets[:, c] == 0)).sum()
        fp = ((preds[:, c] == 1) & (targets[:, c] == 0)).sum()
        specs.append(tn / (tn + fp + 1e-8))
    out["specificity"] = float(np.mean(specs))
    # AUROC (macro, skip degenerate single-class columns)
    aucs = []
    for c in range(targets.shape[1]):
        if len(np.unique(targets[:, c])) > 1:
            aucs.append(roc_auc_score(targets[:, c], probs[:, c]))
    out["auroc"] = float(np.mean(aucs)) if aucs else float("nan")
    return out


# -----------------------------------------------------------------------------
# Continual-learning metrics (from an accuracy matrix R)
# -----------------------------------------------------------------------------
def continual_metrics(R: np.ndarray) -> Dict[str, float]:
    """R[i, j] = test accuracy on domain j after training through domain i.

    Returns Average Accuracy, Forgetting Measure, Backward & Forward Transfer.
    """
    T = R.shape[0]
    avg_acc = float(np.mean(R[T - 1, :]))                      # final-row mean

    # Forgetting: for each earlier task, best-ever minus final accuracy.
    forgetting = []
    for j in range(T - 1):
        best = np.max(R[:T - 1, j]) if T > 1 else R[T - 1, j]
        forgetting.append(best - R[T - 1, j])
    forget = float(np.mean(forgetting)) if forgetting else 0.0

    # Backward transfer: effect of later learning on earlier tasks.
    bwt = float(np.mean([R[T - 1, j] - R[j, j] for j in range(T - 1)])) if T > 1 else 0.0

    # Forward transfer: accuracy on a task before training on it (vs. R diagonal-1).
    fwt = float(np.mean([R[j - 1, j] for j in range(1, T)])) if T > 1 else 0.0

    return {
        "average_accuracy": avg_acc,
        "forgetting": forget,
        "backward_transfer": bwt,
        "forward_transfer": fwt,
    }


# -----------------------------------------------------------------------------
# Calibration metrics
# -----------------------------------------------------------------------------
def expected_calibration_error(probs: np.ndarray, targets: np.ndarray,
                               n_bins: int = 15) -> float:
    """Multi-label ECE: flatten all (sample, label) confidences into bins."""
    p = probs.flatten()
    y = targets.flatten()
    conf = np.where(p >= 0.5, p, 1 - p)        # confidence of the predicted class
    correct = (p >= 0.5).astype(int) == y
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for b in range(n_bins):
        m = (conf > bins[b]) & (conf <= bins[b + 1])
        if m.sum() > 0:
            ece += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return float(ece)


def brier_score(probs: np.ndarray, targets: np.ndarray) -> float:
    return float(np.mean((probs - targets) ** 2))


# -----------------------------------------------------------------------------
# Explainability faithfulness: insertion / deletion (Petsiuk et al., 2018)
# -----------------------------------------------------------------------------
@torch.no_grad()
def deletion_auc(model, batch, heatmap: torch.Tensor, class_idx: int,
                 domain: int, steps: int = 20) -> float:
    """Lower is better: progressively delete most-important pixels; AUC of prob."""
    model.eval()
    x = batch["image"][:1].clone()
    hm = heatmap[:1].flatten()
    order = torch.argsort(hm, descending=True)      # most important first
    n = order.numel()
    single = {k: (v[:1].clone() if torch.is_tensor(v) else v) for k, v in batch.items()}
    probs = []
    for s in range(steps + 1):
        k = int(n * s / steps)
        masked = x.clone().view(1, 3, -1)
        masked[:, :, order[:k]] = 0.0
        single["image"] = masked.view_as(x)
        probs.append(torch.sigmoid(model(single, domain=domain))[0, class_idx].item())
    return float(np.trapz(probs, dx=1.0 / steps))


def localization_overlap(heatmap: torch.Tensor, mask: torch.Tensor) -> float:
    """IoU-style overlap between thresholded saliency and a ground-truth ROI."""
    hm = (heatmap >= heatmap.mean()).float()
    inter = (hm * mask).sum()
    union = ((hm + mask) >= 1).float().sum()
    return float((inter / (union + 1e-8)).item())


In [ ]:
%%writefile cdime_ai/evaluation/stats.py
"""Statistical significance testing (proposal Section 8).

Provides seed-averaged reporting (mean ± std), paired significance tests
(paired t-test / Wilcoxon with a normality check), McNemar's test for paired
classifier comparison, and bootstrap confidence intervals for AUROC / F1.
"""
from __future__ import annotations

from typing import Dict, List, Sequence, Tuple
import numpy as np
from scipy import stats
from sklearn.metrics import roc_auc_score, f1_score


def mean_std(values: Sequence[float]) -> Tuple[float, float]:
    a = np.asarray(values, dtype=float)
    return float(a.mean()), float(a.std(ddof=1)) if len(a) > 1 else 0.0


def paired_comparison(a: Sequence[float], b: Sequence[float],
                      alpha: float = 0.05) -> Dict:
    """Compare two paired sets of per-seed scores.

    Picks a paired t-test if both samples look normal (Shapiro), else the
    Wilcoxon signed-rank test. Returns the test used, statistic, p-value and
    whether the difference is significant at ``alpha``.
    """
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    diff = a - b
    if len(diff) < 3 or np.allclose(diff, 0):
        return {"test": "none", "p_value": 1.0, "significant": False,
                "mean_diff": float(diff.mean())}
    # Normality of the differences.
    normal = stats.shapiro(diff).pvalue > 0.05 if len(diff) >= 3 else True
    if normal:
        stat, p = stats.ttest_rel(a, b)
        test = "paired_t"
    else:
        stat, p = stats.wilcoxon(a, b)
        test = "wilcoxon"
    return {"test": test, "statistic": float(stat), "p_value": float(p),
            "significant": bool(p < alpha), "mean_diff": float(diff.mean())}


def mcnemar_test(y_true: np.ndarray, pred_a: np.ndarray, pred_b: np.ndarray,
                 alpha: float = 0.05) -> Dict:
    """McNemar's test on paired correct/incorrect outcomes of two models."""
    y_true, pred_a, pred_b = map(lambda x: np.asarray(x).flatten(), (y_true, pred_a, pred_b))
    correct_a = pred_a == y_true
    correct_b = pred_b == y_true
    b01 = int(np.sum(correct_a & ~correct_b))   # a right, b wrong
    b10 = int(np.sum(~correct_a & correct_b))   # a wrong, b right
    if b01 + b10 == 0:
        return {"test": "mcnemar", "p_value": 1.0, "significant": False, "b01": b01, "b10": b10}
    # Exact binomial (robust for small/large discordant counts).
    p = stats.binomtest(min(b01, b10), b01 + b10, 0.5).pvalue
    return {"test": "mcnemar", "p_value": float(p), "significant": bool(p < alpha),
            "b01": b01, "b10": b10}


def bootstrap_ci(probs: np.ndarray, targets: np.ndarray, metric: str = "auroc",
                 n_boot: int = 1000, alpha: float = 0.05, seed: int = 0) -> Dict:
    """Bootstrap percentile CI for macro AUROC or macro F1."""
    rng = np.random.default_rng(seed)
    n = len(targets)
    scores = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        p, t = probs[idx], targets[idx]
        try:
            if metric == "auroc":
                vals = [roc_auc_score(t[:, c], p[:, c])
                        for c in range(t.shape[1]) if len(np.unique(t[:, c])) > 1]
                s = np.mean(vals) if vals else np.nan
            else:
                s = f1_score(t, (p >= 0.5).astype(int), average="macro", zero_division=0)
            if not np.isnan(s):
                scores.append(s)
        except ValueError:
            continue
    scores = np.array(scores)
    lo, hi = np.percentile(scores, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return {"metric": metric, "mean": float(scores.mean()),
            "ci_low": float(lo), "ci_high": float(hi), "n_boot": len(scores)}


In [ ]:
%%writefile cdime_ai/evaluation/reporting.py
"""Publication-ready reporting: ablation bar charts and LaTeX tables.

Turns the dicts produced by ``run_continual`` and ``run_ablation_study`` into
(a) matplotlib figures and (b) LaTeX ``tabular`` strings you can paste straight
into a paper. Significance vs. the full model is annotated with stars
(``*`` p<0.05, ``**`` p<0.01, ``***`` p<0.001).
"""
from __future__ import annotations

from typing import Dict, List, Optional


def _stars(p: Optional[float]) -> str:
    if p is None:
        return ""
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return ""


# -----------------------------------------------------------------------------
# Plots
# -----------------------------------------------------------------------------
def ablation_bar_chart(summary: Dict, metric: str = "average_accuracy",
                       title: Optional[str] = None, ax=None,
                       annotate_sig: bool = True):
    """Horizontal bar chart of one metric across variants (mean ± std).

    Returns the matplotlib Axes. The full model is highlighted; bars are sorted
    by mean so the contribution of each ablated component is visually obvious.
    """
    import numpy as np
    import matplotlib.pyplot as plt

    names = list(summary.keys())
    means = [summary[n].get(metric, {}).get("mean", float("nan")) for n in names]
    stds = [summary[n].get(metric, {}).get("std", 0.0) for n in names]
    sig_key = "sig_avg_acc" if metric == "average_accuracy" else "sig_auroc"
    pvals = [summary[n].get(sig_key, {}).get("p_value") for n in names]

    order = sorted(range(len(names)), key=lambda i: (means[i] if means[i] == means[i] else -1))
    names = [names[i] for i in order]
    means = [means[i] for i in order]
    stds = [stds[i] for i in order]
    pvals = [pvals[i] for i in order]

    if ax is None:
        _, ax = plt.subplots(figsize=(9, max(3, 0.5 * len(names))))
    colors = ["#2c7fb8" if "full" in n else "#a6bddb" for n in names]
    bars = ax.barh(names, means, xerr=stds, color=colors, capsize=3)
    ax.set_xlabel(metric.replace("_", " ").title())
    ax.set_title(title or f"Ablation: {metric.replace('_', ' ').title()}")
    if annotate_sig:
        for b, m, p in zip(bars, means, pvals):
            s = _stars(p)
            if s:
                ax.text(b.get_width() + (max(means) * 0.01 if means else 0.01),
                        b.get_y() + b.get_height() / 2, s, va="center", fontsize=11)
    ax.grid(axis="x", alpha=0.3)
    return ax


def plot_r_matrix(R, domains: List[str], ax=None):
    """Heatmap of the continual-learning accuracy matrix R[i, j]."""
    import numpy as np
    import matplotlib.pyplot as plt
    R = np.array(R)
    if ax is None:
        _, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(R, cmap="viridis", vmin=0, vmax=1)
    ax.set_xticks(range(len(domains))); ax.set_xticklabels(domains, rotation=30, ha="right")
    ax.set_yticks(range(len(domains))); ax.set_yticklabels([f"after {d}" for d in domains])
    for i in range(R.shape[0]):
        for j in range(R.shape[1]):
            ax.text(j, i, f"{R[i, j]:.2f}", ha="center", va="center",
                    color="white" if R[i, j] < 0.6 else "black", fontsize=9)
    ax.set_title("Accuracy matrix R[i, j]")
    import matplotlib.pyplot as plt
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    return ax


# -----------------------------------------------------------------------------
# LaTeX tables
# -----------------------------------------------------------------------------
def ablation_to_latex(summary: Dict,
                      metrics: List[str] = ("average_accuracy", "forgetting", "auroc", "f1", "ece"),
                      caption: str = "Ablation study (mean$\\pm$std over seeds). "
                                     "Significance vs.\\ CDIME-AI (full): "
                                     "$^{*}p{<}0.05$, $^{**}p{<}0.01$, $^{***}p{<}0.001$.",
                      label: str = "tab:ablation") -> str:
    """Return a full LaTeX ``table`` (booktabs) for the ablation summary."""
    headers = {
        "average_accuracy": "Avg.\\ Acc.", "forgetting": "Forget.$\\downarrow$",
        "auroc": "AUROC", "f1": "F1", "ece": "ECE$\\downarrow$", "brier": "Brier$\\downarrow$",
        "backward_transfer": "BWT", "forward_transfer": "FWT", "accuracy": "Acc.",
    }
    cols = "l" + "c" * len(metrics)
    lines = [
        "\\begin{table}[t]", "\\centering",
        f"\\caption{{{caption}}}", f"\\label{{{label}}}",
        f"\\begin{{tabular}}{{{cols}}}", "\\toprule",
        "Variant & " + " & ".join(headers.get(m, m) for m in metrics) + " \\\\",
        "\\midrule",
    ]
    for name in summary:
        row = summary[name]
        cells = []
        for m in metrics:
            d = row.get(m, {})
            mean, std = d.get("mean", float("nan")), d.get("std", 0.0)
            star = ""
            if m == "average_accuracy":
                star = _stars(row.get("sig_avg_acc", {}).get("p_value"))
            elif m == "auroc":
                star = _stars(row.get("sig_auroc", {}).get("p_value"))
            cells.append(f"{mean:.3f}$\\pm${std:.3f}{('$^{'+star+'}$') if star else ''}")
        safe = name.replace("_", "\\_")
        bold = "\\textbf{" + safe + "}" if "full" in name else safe
        lines.append(bold + " & " + " & ".join(cells) + " \\\\")
    lines += ["\\bottomrule", "\\end{tabular}", "\\end{table}"]
    return "\n".join(lines)


def continual_to_latex(continual: Dict, bootstrap_auroc: Optional[Dict] = None,
                       caption: str = "Continual-learning performance of CDIME-AI "
                                      "across the CheXpert $\\to$ MIMIC-CXR $\\to$ VinDr-CXR sequence.",
                       label: str = "tab:continual") -> str:
    """LaTeX table for the headline continual-learning metrics."""
    rows = [
        ("Average Accuracy", continual.get("average_accuracy")),
        ("Forgetting $\\downarrow$", continual.get("forgetting")),
        ("Backward Transfer", continual.get("backward_transfer")),
        ("Forward Transfer", continual.get("forward_transfer")),
    ]
    lines = [
        "\\begin{table}[t]", "\\centering",
        f"\\caption{{{caption}}}", f"\\label{{{label}}}",
        "\\begin{tabular}{lc}", "\\toprule", "Metric & Value \\\\", "\\midrule",
    ]
    for name, v in rows:
        lines.append(f"{name} & {v:.3f} \\\\")
    if bootstrap_auroc:
        lines.append(f"AUROC (95\\% CI) & {bootstrap_auroc['mean']:.3f} "
                     f"[{bootstrap_auroc['ci_low']:.3f}, {bootstrap_auroc['ci_high']:.3f}] \\\\")
    lines += ["\\bottomrule", "\\end{tabular}", "\\end{table}"]
    return "\n".join(lines)


In [ ]:
%%writefile cdime_ai/engine/__init__.py
"""CDIME-AI engine subpackage."""


In [ ]:
%%writefile cdime_ai/engine/trainer.py
"""Single-domain trainer with EWC, replay rehearsal and causal (IRM+V-REx) loss.

This is the per-phase workhorse of the continual runner. It mixes replay
exemplars into each batch so that (a) past domains are rehearsed against
forgetting and (b) batches span multiple environments, which is required for the
IRM / V-REx causal penalties to do anything.
"""
from __future__ import annotations

from typing import Optional
import torch
import torch.nn as nn
import torch.nn.functional as F

from ..config import Config
from ..utils import move_batch, get_logger
from ..causal.irm import causal_objective
from ..continual.ewc import EWC
from ..continual.replay import ReplayMemory

logger = get_logger()


def build_optimizer(model: nn.Module, cfg: Config):
    """Separate (smaller) LR group for the pretrained text encoder."""
    text_params, other_params = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        (text_params if "text_encoder" in n else other_params).append(p)
    groups = [{"params": other_params, "lr": cfg.train.lr}]
    if text_params:
        groups.append({"params": text_params, "lr": cfg.train.text_lr})
    return torch.optim.AdamW(groups, weight_decay=cfg.train.weight_decay)


def _merge_batches(cur: dict, replay: Optional[dict], device: str) -> dict:
    cur = move_batch(cur, device)
    if replay is None:
        return cur
    replay = move_batch(replay, device)
    out = {}
    for k in cur:
        if torch.is_tensor(cur[k]):
            out[k] = torch.cat([cur[k], replay[k]], dim=0)
        else:
            out[k] = cur[k]
    return out


def train_one_domain(model, loader, cfg: Config, domain_idx: int,
                     ewc: Optional[EWC] = None,
                     replay: Optional[ReplayMemory] = None) -> None:
    device = cfg.device
    model.to(device)
    model.set_domain(domain_idx)
    model.train()
    opt = build_optimizer(model, cfg)
    amp_on = cfg.train.amp and device == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=amp_on)

    use_irm = cfg.train.use_irm
    use_vrex = cfg.train.use_causal_reg
    replay_n = int(cfg.train.batch_size * cfg.train.replay_ratio)

    for epoch in range(cfg.train.epochs_per_domain):
        # IRM warm-up: ramp the penalty in after a few epochs for stability.
        irm_w = cfg.train.irm_lambda if epoch >= cfg.train.irm_anneal_epochs else 0.0
        running = {"bce": 0.0, "irm": 0.0, "vrex": 0.0, "ewc": 0.0}
        nb = 0
        opt.zero_grad()
        for step, batch in enumerate(loader):
            rep = replay.sample(replay_n) if (replay is not None and not replay.is_empty()) else None
            merged = _merge_batches(batch, rep, device)

            with torch.amp.autocast("cuda", enabled=amp_on):
                logits = model(merged, domain=domain_idx)
                bce = F.binary_cross_entropy_with_logits(logits, merged["label"])
                loss = bce

                # IRM / V-REx need float32 grads of the dummy scalar; compute
                # outside autocast for numerical stability.
            if use_irm or use_vrex:
                logits_f = logits.float()
                irm_t, vrex_t = causal_objective(
                    logits_f, merged["label"], merged["domain"],
                    irm_lambda=(irm_w if use_irm else 0.0),
                    vrex_lambda=(cfg.train.causal_reg_lambda if use_vrex else 0.0))
                loss = loss + irm_t + vrex_t
                running["irm"] += float(irm_t.detach())
                running["vrex"] += float(vrex_t.detach())

            if ewc is not None and cfg.train.use_ewc:
                ewc_pen = ewc.penalty(model)
                loss = loss + ewc_pen
                running["ewc"] += float(ewc_pen.detach())

            loss = loss / cfg.train.grad_accum_steps
            scaler.scale(loss).backward()

            if (step + 1) % cfg.train.grad_accum_steps == 0:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.train.max_grad_norm)
                scaler.step(opt)
                scaler.update()
                opt.zero_grad()

            running["bce"] += float(bce.detach())
            nb += 1

        logger.info(
            f"  [domain {domain_idx} epoch {epoch + 1}/{cfg.train.epochs_per_domain}] "
            f"bce={running['bce']/max(nb,1):.4f} irm={running['irm']/max(nb,1):.4f} "
            f"vrex={running['vrex']/max(nb,1):.4f} ewc={running['ewc']/max(nb,1):.2f}")


In [ ]:
%%writefile cdime_ai/engine/continual_runner.py
"""Domain-incremental orchestration (proposal Section 6, Experimental Protocol).

Phase 1: train on CheXpert.
Phase 2: incrementally update on MIMIC-CXR (no retrain from scratch).
Phase 3: incrementally update on VinDr-CXR.

After every phase we evaluate on *all* domains seen so far to populate the
accuracy matrix R used for continual-learning metrics, and we consolidate EWC +
fill the replay buffer from the just-finished domain.
"""
from __future__ import annotations

from typing import Dict, List, Optional
import numpy as np

from ..config import Config
from ..models.cdime_model import CDIMEModel, AblationFlags
from ..continual.ewc import EWC
from ..continual.replay import ReplayMemory
from ..uncertainty.temperature_scaling import TemperatureScaler
from ..evaluation.metrics import (
    collect_predictions, diagnostic_metrics, continual_metrics,
    expected_calibration_error, brier_score,
)
from ..utils import get_logger, set_seed
from .trainer import train_one_domain

logger = get_logger()


def run_continual(cfg: Config, loaders: Dict[str, Dict], tokenizer,
                  ablation: Optional[AblationFlags] = None,
                  calibrate: bool = True) -> Dict:
    """Run the full continual sequence; return metrics + the trained model."""
    set_seed(cfg.seed)
    ablation = ablation or AblationFlags(
        use_adapters=cfg.train.use_adapters)
    model = CDIMEModel(cfg, ablation).to(cfg.device)

    ewc = EWC(cfg.train.ewc_lambda) if cfg.train.use_ewc else None
    replay = ReplayMemory(cfg.train.replay_size) if cfg.train.use_replay else None

    domains = cfg.domains
    T = len(domains)
    R = np.zeros((T, T))                   # R[i, j]: acc on j after training i
    per_stage: List[Dict] = []
    temperatures: List[float] = []

    for i, domain in enumerate(domains):
        logger.info(f"=== Phase {i + 1}: training on {domain} ===")
        train_one_domain(model, loaders[domain]["train"], cfg, i, ewc, replay)

        # Optional post-hoc calibration on the current domain's val split.
        temp = 1.0
        if calibrate:
            scaler = TemperatureScaler()
            temp = scaler.fit(model, loaders[domain]["val"], i, cfg.device)
        temperatures.append(temp)

        # Evaluate on every domain seen so far (and future ones for FWT).
        stage_metrics = {}
        for j, eval_domain in enumerate(domains):
            probs, targets = collect_predictions(
                model, loaders[eval_domain]["test"], domain=i, device=cfg.device)
            dm = diagnostic_metrics(probs, targets)
            R[i, j] = dm["accuracy"]
            if j <= i:
                dm["ece"] = expected_calibration_error(probs, targets)
                dm["brier"] = brier_score(probs, targets)
                stage_metrics[eval_domain] = dm
        per_stage.append(stage_metrics)
        logger.info(f"  After {domain}: "
                    + ", ".join(f"{d}={R[i, j]:.3f}" for j, d in enumerate(domains)))

        # Consolidate knowledge of the finished domain.
        if ewc is not None:
            ewc.consolidate(model, loaders[domain]["train"], cfg.device, i)
        if replay is not None:
            replay.add_loader(loaders[domain]["train"], i)

    cl = continual_metrics(R)
    logger.info(f"Continual metrics: {cl}")

    return {
        "R_matrix": R.tolist(),
        "continual": cl,
        "per_stage": per_stage,
        "temperatures": temperatures,
        "final_diagnostic": per_stage[-1],
        "model": model,
    }


In [ ]:
%%writefile cdime_ai/ablation.py
"""Ablation study configurations (proposal Section 7) and the experiment driver.

Each ablation toggles one component and is run over multiple seeds. We report
mean ± std for every metric and run the paired significance tests from
``evaluation.stats`` against the full CDIME-AI model.
"""
from __future__ import annotations

from copy import deepcopy
from typing import Callable, Dict, List, Tuple
import numpy as np

from .config import Config
from .models.cdime_model import AblationFlags
from .engine.continual_runner import run_continual
from .evaluation.stats import mean_std, paired_comparison
from .utils import get_logger

logger = get_logger()


def _cfg(base: Config, **train_overrides) -> Config:
    c = deepcopy(base)
    for k, v in train_overrides.items():
        setattr(c.train, k, v)
    return c


def build_variants(base: Config) -> Dict[str, Tuple[Config, AblationFlags]]:
    """Return {name: (config, ablation_flags)} for the full study."""
    full_flags = AblationFlags(use_text=True, use_cross_attention=True,
                               use_adapters=True, enable_mc_dropout=True)
    return {
        # Reference model
        "CDIME-AI (full)": (deepcopy(base), full_flags),

        # Baselines named in Section 6
        "Baseline: multimodal, no continual": (
            _cfg(base, use_ewc=False, use_replay=False, use_adapters=False,
                 use_irm=False, use_causal_reg=False), full_flags),
        "Baseline: continual, no causal": (
            _cfg(base, use_irm=False, use_causal_reg=False), full_flags),

        # A1 remove causal learning
        "A1: no causal": (_cfg(base, use_irm=False, use_causal_reg=False), full_flags),
        # A2 remove domain-incremental learning
        "A2: no domain-incremental": (
            _cfg(base, use_ewc=False, use_replay=False, use_adapters=False),
            AblationFlags(use_text=True, use_cross_attention=True,
                          use_adapters=False, enable_mc_dropout=True)),
        # A3 remove explainability (no effect on predictive metrics; tracked separately)
        "A3: no explainability": (deepcopy(base), full_flags),
        # A4 remove uncertainty estimation (no MC-dropout / temp scaling)
        "A4: no uncertainty": (deepcopy(base),
                               AblationFlags(use_text=True, use_cross_attention=True,
                                             use_adapters=True, enable_mc_dropout=False)),
        # A5 image-only
        "A5: image-only": (deepcopy(base),
                           AblationFlags(use_text=False, use_cross_attention=False,
                                         use_adapters=True, enable_mc_dropout=True)),
        # A6 concat instead of cross-attention
        "A6: concat fusion": (deepcopy(base),
                              AblationFlags(use_text=True, use_cross_attention=False,
                                            use_adapters=True, enable_mc_dropout=True)),
        # A7 each CL strategy individually
        "A7a: EWC only": (_cfg(base, use_replay=False, use_adapters=False), full_flags),
        "A7b: Replay only": (
            _cfg(base, use_ewc=False, use_adapters=False),
            AblationFlags(use_text=True, use_cross_attention=True,
                          use_adapters=False, enable_mc_dropout=True)),
        "A7c: Adapters only": (_cfg(base, use_ewc=False, use_replay=False), full_flags),
    }


def run_seeded(cfg: Config, flags: AblationFlags, tokenizer, seeds: List[int],
               loader_builder: Callable) -> Dict[str, List[float]]:
    """Run one variant across seeds; collect per-seed scalar metrics."""
    keys = ["average_accuracy", "forgetting", "backward_transfer", "forward_transfer"]
    diag_keys = ["accuracy", "f1", "auroc", "ece", "brier"]
    acc: Dict[str, List[float]] = {k: [] for k in keys + diag_keys}

    for s in seeds:
        c = deepcopy(cfg)
        c.seed = s
        calibrate = flags.enable_mc_dropout  # A4 disables calibration too
        loaders = loader_builder(c, tokenizer)
        res = run_continual(c, loaders, tokenizer, ablation=flags, calibrate=calibrate)
        for k in keys:
            acc[k].append(res["continual"][k])
        # Average the final-stage diagnostic metrics over seen domains.
        final = res["final_diagnostic"]
        for dk in diag_keys:
            vals = [m[dk] for m in final.values() if dk in m and not np.isnan(m.get(dk, np.nan))]
            acc[dk].append(float(np.mean(vals)) if vals else float("nan"))
    return acc


def run_ablation_study(base: Config, tokenizer, loader_builder: Callable,
                       seeds: List[int] = (42, 43, 44)) -> Dict:
    """Run every variant over seeds and compare each against the full model."""
    variants = build_variants(base)
    raw: Dict[str, Dict[str, List[float]]] = {}
    for name, (cfg, flags) in variants.items():
        logger.info(f"\n########## Variant: {name} ##########")
        raw[name] = run_seeded(cfg, flags, tokenizer, list(seeds), loader_builder)

    # Summarise mean ± std and significance vs. full model.
    ref = raw["CDIME-AI (full)"]
    table: Dict[str, Dict] = {}
    for name, metrics in raw.items():
        row = {}
        for k, vals in metrics.items():
            m, sd = mean_std([v for v in vals if not np.isnan(v)])
            row[k] = {"mean": m, "std": sd}
        # Significance on average_accuracy and auroc vs the full model.
        if name != "CDIME-AI (full)":
            row["sig_avg_acc"] = paired_comparison(
                ref["average_accuracy"], metrics["average_accuracy"])
            row["sig_auroc"] = paired_comparison(ref["auroc"], metrics["auroc"])
        table[name] = row
    return {"raw": raw, "summary": table, "seeds": list(seeds)}


In [ ]:
# Make the freshly-written package importable in this runtime
import importlib, sys
for k in [m for m in list(sys.modules) if m.startswith('cdime_ai')]:
    del sys.modules[k]
import cdime_ai
print('cdime_ai version', cdime_ai.__version__, '— package built OK')

## 2. Configure (T4-friendly)
`Config()` is tuned for a 16 GB T4. Raise `epochs_per_domain` / `train_per_domain`
for full Q1-scale results when you have more time/compute.

In [ ]:
from cdime_ai.config import Config
from cdime_ai.tokenizer import get_tokenizer
from cdime_ai.data.loaders import build_domain_loaders

cfg = Config()
cfg.data.mode = 'synthetic'        # 'real' to use on-disk CheXpert/MIMIC/VinDr
cfg.train.epochs_per_domain = 4    # raise to 8-15 for the paper
cfg.train.batch_size = 8
cfg.data.train_per_domain = 600
print('device:', cfg.device, '| domains:', cfg.domains)
print('labels:', cfg.labels)

tokenizer = get_tokenizer(cfg)     # ClinicalBERT (auto-fallback to DistilBERT)
loaders = build_domain_loaders(cfg, tokenizer)
print('loaders ready for', list(loaders.keys()))

## 3. Run the continual protocol (Phase 1 → 2 → 3)

In [ ]:
from cdime_ai.engine.continual_runner import run_continual
results = run_continual(cfg, loaders, tokenizer)
model = results.pop('model')
print('\nContinual-learning metrics:')
for k, v in results['continual'].items():
    print(f'  {k:20s}: {v:.4f}')

In [ ]:
import numpy as np, pandas as pd
R = np.array(results['R_matrix'])
df = pd.DataFrame(R, index=[f'after {d}' for d in cfg.domains], columns=cfg.domains)
print('Accuracy matrix R[i,j] = test acc on domain j after training through i:')
display(df.round(3))

import matplotlib.pyplot as plt
from cdime_ai.evaluation.reporting import plot_r_matrix
plot_r_matrix(results['R_matrix'], cfg.domains); plt.tight_layout(); plt.show()

## 4. Bootstrap confidence intervals (AUROC / F1)

In [ ]:
from cdime_ai.evaluation.metrics import collect_predictions
from cdime_ai.evaluation.stats import bootstrap_ci
last = len(cfg.domains) - 1
probs, targets = collect_predictions(model, loaders[cfg.domains[last]]['test'], domain=last, device=cfg.device)
print('AUROC:', bootstrap_ci(probs, targets, 'auroc', n_boot=1000, seed=cfg.seed))
print('F1   :', bootstrap_ci(probs, targets, 'f1',    n_boot=1000, seed=cfg.seed))

## 5. Explainability — Grad-CAM, attention evidence, counterfactual + uncertainty

In [ ]:
import matplotlib.pyplot as plt, torch
from cdime_ai.utils import move_batch
from cdime_ai.explain.gradcam import GradCAM
from cdime_ai.explain.attention import image_to_text_attention, top_text_evidence
from cdime_ai.explain.counterfactual import generate_counterfactual
from cdime_ai.uncertainty.mc_dropout import mc_dropout_predict

batch = move_batch(next(iter(loaders[cfg.domains[last]]['test'])), cfg.device)
mean_p, std_p, ent = mc_dropout_predict(model, batch, last, n_samples=20)
cls = int(mean_p[0].argmax())
cam = GradCAM(model)(batch, class_idx=cls, domain=last)

img = batch['image'][0].cpu(); img = (img - img.min())/(img.max()-img.min())
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].imshow(img.permute(1,2,0)); ax[0].set_title('Input CXR'); ax[0].axis('off')
ax[1].imshow(cam[0].cpu(), cmap='jet'); ax[1].set_title(f'Grad-CAM: {cfg.labels[cls]}'); ax[1].axis('off')
ax[2].imshow(img.permute(1,2,0)); ax[2].imshow(cam[0].cpu(), cmap='jet', alpha=0.5)
ax[2].set_title('Overlay'); ax[2].axis('off'); plt.tight_layout(); plt.show()

attn = image_to_text_attention(model, batch, domain=last)
print('Predicted top class:', cfg.labels[cls], '| predictive entropy:', float(ent[0]))
print('MC-Dropout std (epistemic):', np.round(std_p[0].cpu().numpy(), 3))
print('Top report evidence:', top_text_evidence(attn[0] if attn is not None else None, batch['input_ids'][0], tokenizer, 5))
cf, delta, p0, p1 = generate_counterfactual(model, batch, class_idx=cls, domain=last)
print(f'Counterfactual {cfg.labels[cls]}: p {p0:.2f} -> {p1:.2f}  (L2 change {delta.norm():.2f})')

## 6. Therapeutic decision-support report

In [ ]:
from cdime_ai.decision.therapeutic import make_decision
ev = top_text_evidence(attn[0] if attn is not None else None, batch['input_ids'][0], tokenizer, 3)
report = make_decision(cfg.labels, mean_p[0].cpu().numpy(), std_p[0].cpu().numpy(),
                       evidence=[f'{t} ({w:.2f})' for t, w in ev])
print(report.to_text())

## 7. Ablation study + significance tests (optional, longer)
Runs all baselines + ablations A1–A7 over several seeds and runs paired
t-test / Wilcoxon vs. the full model. Use more seeds for the paper.

In [ ]:
from cdime_ai.ablation import run_ablation_study

# Pretty-printer for the ablation table (defined inline):
def _print_ablation_table(summary):
    print('%-38s %14s %14s %8s %8s' % ('Variant','AvgAcc','Forget','AUROC','p(acc)'))
    for name, row in summary.items():
        aa=row.get('average_accuracy',{}); fg=row.get('forgetting',{}); au=row.get('auroc',{}); sig=row.get('sig_avg_acc',{})
        print('%-38s %6.3f+/-%.3f %6.3f+/-%.3f %6.3f %8s' % (
            name[:38], aa.get('mean',0), aa.get('std',0), fg.get('mean',0), fg.get('std',0),
            au.get('mean',0), (f"{sig.get('p_value',float('nan')):.3f}" if sig else 'ref')))

study = run_ablation_study(cfg, tokenizer, build_domain_loaders, seeds=[42, 43])
_print_ablation_table(study['summary'])

## 8. Publication figures & LaTeX tables
Turn the results above into paper-ready outputs: a sorted ablation bar chart
(significance stars vs. the full model) and `booktabs` LaTeX tables saved to
`paper_tables/` — paste straight into your manuscript.

In [ ]:
import os, matplotlib.pyplot as plt
from cdime_ai.evaluation.reporting import (
    ablation_bar_chart, ablation_to_latex, continual_to_latex)

os.makedirs('paper_tables', exist_ok=True)

# --- Ablation bar charts (average accuracy + forgetting) ---
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
ablation_bar_chart(study['summary'], 'average_accuracy', ax=axes[0])
ablation_bar_chart(study['summary'], 'forgetting', ax=axes[1])
plt.tight_layout(); plt.savefig('paper_tables/ablation_bars.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# --- LaTeX tables ---
ablation_tex = ablation_to_latex(study['summary'])
continual_tex = continual_to_latex(
    results['continual'],
    bootstrap_ci(probs, targets, 'auroc', n_boot=1000, seed=cfg.seed))

open('paper_tables/ablation_table.tex', 'w').write(ablation_tex)
open('paper_tables/continual_table.tex', 'w').write(continual_tex)
print(continual_tex)
print('\n' + '='*70 + '\n')
print(ablation_tex)
print('\nSaved to paper_tables/  (download via the Colab Files panel).')

In [ ]:
# Optional: zip the paper outputs for one-click download in Colab
import shutil
shutil.make_archive('cdime_paper_outputs', 'zip', 'paper_tables')
try:
    from google.colab import files
    files.download('cdime_paper_outputs.zip')
except Exception:
    print('Outputs in paper_tables/ and cdime_paper_outputs.zip')

## 9. Using the REAL datasets
The whole pipeline is dataset-agnostic. To switch from synthetic to real data:

1. Download **subsets** of CheXpert / MIMIC-CXR / VinDr-CXR (the full sets cannot
   fit on free Colab) and arrange a small CSV + image folder per domain.
2. Edit `REAL_PATHS` inside the `cdime_ai/data/datasets.py` cell above to point at
   them (columns: standard CheXpert labels; MIMIC also needs a `report` column).
3. Set `cfg.data.mode = 'real'` and rerun from Section 2.

Expected CSV columns are documented in the `datasets.py` cell. For Q1 results,
run ≥5 seeds and scale up `epochs_per_domain` / `train_per_domain`.